In [2]:
import pandas as pd
import numpy as np
import gc

In [3]:
df = pd.read_parquet('df_preprocessed.parquet')

In [4]:
def downcast(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    for i, t in enumerate(types):
        if 'int' in str(t):
            if df[cols[i]].min() > np.iinfo(np.int8).min and df[cols[i]].max() < np.iinfo(np.int8).max:
                df[cols[i]] = df[cols[i]].astype(np.int8)
            elif df[cols[i]].min() > np.iinfo(np.int16).min and df[cols[i]].max() < np.iinfo(np.int16).max:
                df[cols[i]] = df[cols[i]].astype(np.int16)
            elif df[cols[i]].min() > np.iinfo(np.int32).min and df[cols[i]].max() < np.iinfo(np.int32).max:
                df[cols[i]] = df[cols[i]].astype(np.int32)
        elif 'float' in str(t):
            if df[cols[i]].min() > np.finfo(np.float16).min and df[cols[i]].max() < np.finfo(np.float16).max:
                df[cols[i]] = df[cols[i]].astype(np.float16)
            elif df[cols[i]].min() > np.finfo(np.float32).min and df[cols[i]].max() < np.finfo(np.float32).max:
                df[cols[i]] = df[cols[i]].astype(np.float32)
        elif t == object:
            df[cols[i]] = df[cols[i]].astype('category')
    return df

In [5]:
grouped_sales = df.groupby(['item_id', 'store_id'])['sales']

for lag in [28, 29, 30]:
  df[f'lag_{lag}'] = grouped_sales.shift(lag).astype(np.float16)

del grouped_sales
gc.collect()

0

In [6]:
shifted_sales = df.groupby(['item_id', 'store_id'])['sales'].shift(28).astype(np.float16)

for window in [7, 30, 90, 180]:
    df[f'rolling_mean_{window}'] = shifted_sales.rolling(window).mean().astype(np.float16)

for window in [7, 30]:
    df[f'rolling_std_{window}'] = shifted_sales.rolling(window).std().astype(np.float16)

del shifted_sales
gc.collect()

0

In [7]:
df['price_change'] = df['sell_price'] - df.groupby(['item_id', 'store_id'])['sell_price'].shift(7)  # no leakage since its not target

df['relative_price'] = df['sell_price'] / df.groupby(['item_id'])['sell_price'].transform('mean')

df = downcast(df)

In [8]:
df.head()

,item_id,dept_id,cat_id,store_id,state_id,d,day,wday,wno,month,...,lag_29,lag_30,rolling_mean_7,rolling_mean_30,rolling_mean_90,rolling_mean_180,rolling_std_7,rolling_std_30,price_change,relative_price
0,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,29,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.923828
1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,2,30,2,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.923828
2,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,3,31,3,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.923828
3,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,4,1,4,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.923828
4,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,5,2,5,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.923828


In [9]:
df.dtypes

,0
item_id,category
dept_id,category
cat_id,category
store_id,category
state_id,category
d,int16
day,int8
wday,int8
wno,int8
month,int8


In [10]:
dtypes = df.dtypes.tolist()
cols = df.columns.tolist()
for i in range(len(cols)):
  if dtypes[i] == 'category':
    df[cols[i]] = df[cols[i]].cat.codes

df = downcast(df)

In [11]:
df.columns

Index(['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'day',
       'wday', 'wno', 'month', 'year', 'has_event', 'event_type_1',
       'event_type_2', 'snap', 'sell_price', 'sales', 'lag_28', 'lag_29',
       'lag_30', 'rolling_mean_7', 'rolling_mean_30', 'rolling_mean_90',
       'rolling_mean_180', 'rolling_std_7', 'rolling_std_30', 'price_change',
       'relative_price'],
      dtype='object')

In [12]:
cols=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'day', 'wday', 'wno', 'month', 'year', 'has_event', 'event_type_1', 'event_type_2', 'snap', 'sell_price', 'price_change', 'relative_price', 'lag_28', 'lag_29', 'lag_30', 'rolling_mean_7', 'rolling_mean_30', 'rolling_mean_90', 'rolling_mean_180', 'rolling_std_7', 'rolling_std_30', 'sales']
df = df[cols]

In [13]:
df.head()

,item_id,dept_id,cat_id,store_id,state_id,d,day,wday,wno,month,...,lag_28,lag_29,lag_30,rolling_mean_7,rolling_mean_30,rolling_mean_90,rolling_mean_180,rolling_std_7,rolling_std_30,sales
0,0,0,0,0,0,1,29,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
1,0,0,0,0,0,2,30,2,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,0,0,0,0,0,3,31,3,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,0,0,0,0,0,4,1,4,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,0,0,0,0,0,5,2,5,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4


In [14]:
df.dtypes

,0
item_id,int16
dept_id,int8
cat_id,int8
store_id,int8
state_id,int8
d,int16
day,int8
wday,int8
wno,int8
month,int8


In [15]:
df.to_parquet('df_featured.parquet', index=False)